# Understanding Chunking Strategies for LLMs and RAG

In Large Language Model (LLM) applications, particularly Retrieval-Augmented Generation (RAG), we often need to process documents that are longer than the model's context window. **Chunking** is the process of breaking down large pieces of text into smaller, manageable segments (chunks).

The strategy you choose directly impacts how well your retrieval system understands context and finds relevant information.

In this notebook, we will explore:
1. Fixed-Size Chunking (Character & Token-based)
2. Sentence-Based Chunking
3. Recursive Character Chunking
4. Semantic Chunking

In [ ]:
# Install required libraries
!pip install -qU langchain langchain-experimental langchain-huggingface nltk

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# Let's define a sample text to chunk
sample_text = """
Turning the device off
To turn off the device, press and hold the Side button. Alternatively, open the quick 
settings panel and tap the Power off icon ( ).
Tap Power off → Power off.
To restart the device, tap Restart → Restart
"""

print(f"Total length of sample text: {len(sample_text)} characters.")

Fixed-Size Chunking

The simplest method. We divide the text into chunks of a specific number of characters or tokens. We often add an **overlap** between chunks so that words or concepts at the edge of a chunk aren't completely lost.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# Split by every 100 characters, with a 20-character overlap
fixed_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=100,
    chunk_overlap=20
)

fixed_chunks = fixed_splitter.split_text(sample_text)

print(f"Number of chunks: {len(fixed_chunks)}\n")
for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()

2. Sentence-Based Chunking

Fixed-size chunking often cuts sentences in half, destroying their grammatical meaning. Sentence-based chunking uses NLP tools (like NLTK or SpaCy) to split text cleanly at sentence boundaries.

## 3. Recursive Character Chunking

This is the recommended default for text in frameworks like LangChain. It tries to split text using a hierarchy of separators (e.g., double newlines `\n\n`, then single newlines `\n`, then spaces ` `, then characters `""`).

This keeps paragraphs together if possible, then sentences, then words, ensuring chunks are as semantically intact as possible while strictly respecting the maximum chunk size.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)

recursive_chunks = recursive_splitter.split_text(sample_text)

print(f"Number of chunks: {len(recursive_chunks)}\n")
for i, chunk in enumerate(recursive_chunks[:4]):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()

## 4. Semantic Chunking

Instead of relying on punctuation or character counts, semantic chunking uses **embedding models** to calculate the mathematical similarity between sentences. It groups adjacent sentences into the same chunk *only* if their meanings are highly related. If there is a sudden drop in similarity (a topic shift), it starts a new chunk.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

# Load a fast, lightweight open-source embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Initialize the Semantic Chunker
# We use the "percentile" breakpoint type, which splits when the difference
# between adjacent sentences exceeds a certain percentile threshold.
semantic_chunker = SemanticChunker(embeddings, breakpoint_threshold_type="percentile")

semantic_chunks = semantic_chunker.split_text(sample_text)

print(f"Number of chunks: {len(semantic_chunks)}\n")
for i, chunk in enumerate(semantic_chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()